# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the [FAIR](https://www.go-fair.org/fair-principles/) principles.

In [ ]:
# Install `mlcroissant` if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This allows you to access and process the dataset in a standardized way.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (access as attribute rather than as a dictionary)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Examine available record sets, their unique `@id` values, and the fields they contain. Referencing by `@id` is essential for precise and reproducible data operations.

In [ ]:
# List all record sets with their @id and fields
print("Available Record Sets and their Fields (by @id):\n")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"Record Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    field_ids = [field.id for field in record_set.fields]
    print(f"  Field @ids: {field_ids}")
    print()
    record_set_ids.append(record_set.id)


## 3. Data Extraction
Load data from one or more record sets, referencing them by `@id`. For each record set, data are loaded into a pandas DataFrame.

In [ ]:
# Load all record sets (by @id) into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading data for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")
if not record_set_ids:
    print("No record sets found in this Croissant schema.")

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate exploratory data analysis on a record set and its numeric field. Please change the `selected_record_set_id` and fields to match the output above if analyzing a different subset.

In [ ]:
# Pick a record set and field for analysis
# For this template, we assume the main record set contains protein abundance and has field @ids accordingly

if record_set_ids:
    selected_record_set_id = record_set_ids[0]  # Change this index if analyzing another set
    df = dataframes[selected_record_set_id]
    print(f"Using record set: {selected_record_set_id}")
    print(f"Available columns: {df.columns.tolist()}")
    # Try to select a likely numeric field: look for typical column names
    # If you know the @id, set numeric_field_id directly
    numeric_candidates = [c for c in df.columns if 'abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower() or 'intensity' in c.lower() or 'coverage' in c.lower()]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        # Fallback: pick the first column
        numeric_field_id = df.columns[0]
    print(f"Numeric field selected (by @id): {numeric_field_id}")

    # Filter out null/NaN and apply threshold
    df_filtered = df.copy()
    # Convert the field to numeric, coerce errors
    df_filtered[numeric_field_id] = pd.to_numeric(df_filtered[numeric_field_id], errors='coerce')

    threshold = df_filtered[numeric_field_id].mean() if not df_filtered[numeric_field_id].isnull().all() else 0
    filtered_df = df_filtered[df_filtered[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    if filtered_df[numeric_field_id].count() > 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    group_candidates = [c for c in df.columns if c != numeric_field_id and ('type' in c.lower() or 'category' in c.lower() or 'sample' in c.lower() or 'group' in c.lower())]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No obvious categorical/group field found for grouping.")
else:
    print("No record set data loaded for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field. This assists in understanding the field's spread and possible anomalies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
else:
    print("No numeric field or data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its Croissant schema, explored available record sets and fields (referenced by their `@id`s), extracted and analyzed numeric fields, and visualized the data. This workflow ensures reproducibility and clear referencing for scientific data analysis.

**Next steps:** You may extend this notebook by performing more advanced analyses, exporting clean data, or integrating the records into further bioinformatics pipelines.